# 💰 RAG for Finance

## The Use Case

Financial analysts and advisors need fast access to:
- Earnings reports and 10-Ks
- Market research and analyst notes
- Regulatory filings (SEC, FINRA)

RAG makes financial research 10x faster.

## What's Special About Finance?

```
1. Numbers matter — exact figures must be preserved
2. Time-sensitivity — Q3 2024 vs Q3 2023 are very different
3. Regulatory constraints — cannot give investment advice
4. Comparisons — often need to compare across companies or periods
```

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Ready!")

Ready!


In [2]:
# Simulated financial document chunks

finance_docs = [
    {"source": "ACME Corp 10-K 2024",       "period": "FY2024", "text": "Total revenue increased 18% year-over-year to $4.2 billion. Growth was driven by cloud services (+34%) and software subscriptions (+22%)."},
    {"source": "ACME Corp 10-K 2024",       "period": "FY2024", "text": "Net income was $620 million, down 8% from the prior year, primarily due to increased R&D investment of $980 million."},
    {"source": "ACME Corp 10-K 2023",       "period": "FY2023", "text": "Total revenue was $3.56 billion. Net income reached $673 million, up 12% from FY2022."},
    {"source": "Analyst Report - BofA",     "period": "Q3 2024", "text": "We maintain a BUY rating on ACME Corp with a price target of $185. Strong cloud growth offsets hardware margin pressure."},
    {"source": "ACME Earnings Call Q3 2024","period": "Q3 2024", "text": "CEO stated: 'We expect cloud revenue to exceed $2 billion in FY2025, representing 48% of total revenue.'"},
    {"source": "Risk Factors 10-K 2024",    "period": "FY2024", "text": "Key risks include intensifying competition in cloud services, macroeconomic slowdown, and potential regulatory changes in data privacy."},
]

texts      = [d["text"] for d in finance_docs]
embeddings = model.encode(texts)
print(f"Indexed {len(finance_docs)} financial document chunks")

Indexed 6 financial document chunks


In [3]:
# Finance search — shows source, period, and relevance score

def finance_search(question, top_k=2):
    q_emb  = model.encode(question)
    scores = cosine_similarity([q_emb], embeddings)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"Query: '{question}'")
    print("-" * 65)
    for rank, idx in enumerate(top_idx, 1):
        doc = finance_docs[idx]
        print(f"  {rank}. [{scores[idx]:.3f}] {doc['source']} ({doc['period']})")
        print(f"     {doc['text']}")
    print()
    print("⚠️  Not investment advice. Always verify with primary source filings.")
    print()

finance_search("What was ACME's revenue growth?")
finance_search("What are the main risks for ACME?")
finance_search("What is the analyst outlook for ACME?")

Query: 'What was ACME's revenue growth?'
-----------------------------------------------------------------
  1. [0.526] ACME Corp 10-K 2024 (FY2024)
     Total revenue increased 18% year-over-year to $4.2 billion. Growth was driven by cloud services (+34%) and software subscriptions (+22%).
  2. [0.505] Analyst Report - BofA (Q3 2024)
     We maintain a BUY rating on ACME Corp with a price target of $185. Strong cloud growth offsets hardware margin pressure.

⚠️  Not investment advice. Always verify with primary source filings.

Query: 'What are the main risks for ACME?'
-----------------------------------------------------------------
  1. [0.360] Risk Factors 10-K 2024 (FY2024)
     Key risks include intensifying competition in cloud services, macroeconomic slowdown, and potential regulatory changes in data privacy.
  2. [0.335] Analyst Report - BofA (Q3 2024)
     We maintain a BUY rating on ACME Corp with a price target of $185. Strong cloud growth offsets hardware margin pressure.

In [4]:
# Filter by time period — crucial in finance

def finance_search_by_period(question, period_filter=None, top_k=3):
    # Filter docs by period if specified
    if period_filter:
        filtered_idx = [i for i, d in enumerate(finance_docs) if period_filter in d["period"]]
    else:
        filtered_idx = list(range(len(finance_docs)))

    if not filtered_idx:
        print(f"No documents found for period: {period_filter}")
        return

    q_emb           = model.encode(question)
    filtered_embs   = embeddings[filtered_idx]
    scores          = cosine_similarity([q_emb], filtered_embs)[0]
    top_local_idx   = np.argsort(scores)[::-1][:top_k]
    top_global_idx  = [filtered_idx[i] for i in top_local_idx]

    label = f" (filtered to {period_filter})" if period_filter else ""
    print(f"Query: '{question}'{label}")
    print("-" * 65)
    for rank, idx in enumerate(top_global_idx, 1):
        doc = finance_docs[idx]
        print(f"  {rank}. [{scores[top_local_idx[rank-1]]:.3f}] {doc['source']} ({doc['period']})")
        print(f"     {doc['text'][:80]}...")
    print()

# Compare: all periods vs just FY2024
finance_search_by_period("What was net income?", period_filter=None)
finance_search_by_period("What was net income?", period_filter="FY2024")

Query: 'What was net income?'
-----------------------------------------------------------------
  1. [0.682] ACME Corp 10-K 2023 (FY2023)
     Total revenue was $3.56 billion. Net income reached $673 million, up 12% from FY...
  2. [0.645] ACME Corp 10-K 2024 (FY2024)
     Net income was $620 million, down 8% from the prior year, primarily due to incre...
  3. [0.359] ACME Corp 10-K 2024 (FY2024)
     Total revenue increased 18% year-over-year to $4.2 billion. Growth was driven by...

Query: 'What was net income?' (filtered to FY2024)
-----------------------------------------------------------------
  1. [0.645] ACME Corp 10-K 2024 (FY2024)
     Net income was $620 million, down 8% from the prior year, primarily due to incre...
  2. [0.359] ACME Corp 10-K 2024 (FY2024)
     Total revenue increased 18% year-over-year to $4.2 billion. Growth was driven by...
  3. [0.149] Risk Factors 10-K 2024 (FY2024)
     Key risks include intensifying competition in cloud services, macroeconomic slow.

## Key Rules for Finance RAG

1. **Always show period** — FY2023 vs FY2024 results are completely different
2. **Preserve exact numbers** — don't summarise financials, show the exact figures
3. **Cite the filing** — 10-K, 10-Q, earnings call — with date
4. **No investment advice** — present data, not recommendations
5. **Filter by recency** — users usually want the latest filing, not old ones